# Ingest circuits.csv file


Step 1: Initialize the setup configuration, allowing access to cloud storage


In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
%run "../01-setup/2.common_functions"

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2025-01")
v_file_date = dbutils.widgets.get("p_file_date")
raw_race_path = f"{raw_folder_path}/{v_file_date}"

Step 2: Import libraries for StructFields

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

Step 3: Read the files using the spark dataframe reader API

In [0]:
circuits_schema = StructType([
    StructField("circuitId", IntegerType(), False),
    StructField("circuitRef", StringType(), True),
    StructField("name", StringType(), True),
    StructField("location", StringType(), True),
    StructField("country", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lng", DoubleType(), True),
    StructField("alt", IntegerType(), True),
    StructField("url", StringType(), True)
])
circuits_df = spark.read.csv(
    f"{raw_race_path}/circuits.csv",
    header=True,
    schema=circuits_schema
)
display(circuits_df)
circuits_df.printSchema()
circuits_df.describe().show()

Step 4: Select the required columns

In [0]:
from pyspark.sql.functions import lit
circuits_selected_df = circuits_df.select("circuitId", "circuitRef", "name", "location", "country", "lat", "lng", "alt")
display(circuits_selected_df)
circuits_renamed_df = circuits_selected_df.withColumnRenamed("circuitId", "circuit_id") \
                                         .withColumnRenamed("circuitRef", "circuit_ref") \
                                         .withColumnRenamed("lat", "latitude") \
                                         .withColumnRenamed("lng", "longitude") \
                                         .withColumnRenamed("alt", "altitude") \
                                         .withColumn("data_source", lit(v_data_source))
display(circuits_renamed_df)

Step 5: Add ingestion date to the dataframe

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
circuits_final_df = add_ingestion_date(circuits_renamed_df)
display(circuits_final_df)

Step 6: Write data to datalake as parquet


In [0]:
circuits_final_df.write.mode("overwrite").parquet(f"{processed_folder_path}/circuits")

In [0]:
df = spark.read.parquet(f"{processed_folder_path}/circuits")
display(df)

In [0]:
dbutils.notebook.exit("Success")